# Naive Bayes Classification — Student Dropout & Academic Success

**Dataset:** Predict Students' Dropout and Academic Success  
**Source:** https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success  
**Size:** 4,424 students | 36 features | 3 target classes (Dropout, Enrolled, Graduate)  
**Features include:** Demographics, application details, 1st & 2nd semester academic performance, socioeconomic indicators (GDP, unemployment, inflation)

In [ ]:
!pip install ucimlrepo imbalanced-learn --quiet
from ucimlrepo import fetch_ucirepo
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
import pkg_resources

packages = [
    "numpy", "pandas", "scikit-learn", "matplotlib",
    "seaborn", "imbalanced-learn", "ucimlrepo"
]
for p in packages:
    try:
        print(f"{p}: {pkg_resources.get_distribution(p).version}")
    except:
        print(f"{p}: not installed")

In [ ]:
dataset = fetch_ucirepo(id=697)
X, y = dataset.data.features, dataset.data.targets

print(f'Shape: {X.shape}  |  Missing values: {X.isnull().sum().sum()}')
print('\nTarget distribution:')
print(y['Target'].value_counts().to_string())

counts = y['Target'].value_counts()
plt.figure(figsize=(6, 3))
bars = plt.bar(counts.index, counts.values,
               color=['#E8593C','#3B8BD4','#3DA87A'], edgecolor='white')
for b in bars:
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+20,
             str(int(b.get_height())), ha='center', fontsize=10)
plt.title('Target class distribution')
plt.ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
# Impute
imputer = SimpleImputer(strategy='mean')
X_imp = imputer.fit_transform(X)

# Encode target
le = LabelEncoder()
y_enc = le.fit_transform(y['Target'])
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

# Feature selection
sel = VarianceThreshold(threshold=0.01)
X_sel = sel.fit_transform(X_imp)
print(f'Features: {X_imp.shape[1]} -> {X_sel.shape[1]} ({X_imp.shape[1]-X_sel.shape[1]} low-variance removed)')

# Scale
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_sel)

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_sc, y_enc, test_size=0.20, random_state=42, stratify=y_enc)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print('Train distribution:', dict(zip(le.classes_, np.bincount(y_train))))
print('Test  distribution:', dict(zip(le.classes_, np.bincount(y_test))))

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

var_smoothing_values = np.logspace(-12, 0, 25)
cv_scores = []

print('Tuning var_smoothing (5-fold CV):')
print(f"{'var_smoothing':>16} {'CV Accuracy':>12} {'Std':>8}")
print('-' * 40)
for vs in var_smoothing_values:
    gnb = GaussianNB(var_smoothing=vs)
    scores = cross_val_score(gnb, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_scores.append(scores.mean())
    print(f'{vs:>16.2e} {scores.mean():>12.4f} {scores.std():>8.4f}')

best_vs = var_smoothing_values[np.argmax(cv_scores)]
print(f'\nBest var_smoothing: {best_vs:.2e}  (CV accuracy: {max(cv_scores):.4f})')

plt.figure(figsize=(8, 3))
plt.semilogx(var_smoothing_values, cv_scores, marker='o', markersize=5,
             color='#3DA87A', linewidth=2)
plt.axvline(best_vs, linestyle='--', color='#E8593C', linewidth=1.5,
            label=f'Best var_smoothing={best_vs:.2e}')
plt.title('5-fold CV accuracy vs var_smoothing (Gaussian Naive Bayes)')
plt.xlabel('var_smoothing'); plt.ylabel('CV Accuracy')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
gnb = GaussianNB(var_smoothing=best_vs)
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)
print(f'Model trained. Predicting on {X_test.shape[0]} test samples.')

In [ ]:
acc    = accuracy_score(y_test, y_pred)
f1_mac = f1_score(y_test, y_pred, average='macro')
f1_wt  = f1_score(y_test, y_pred, average='weighted')

print('=' * 52)
print('        NAIVE BAYES TEST RESULTS')
print('=' * 52)
print(f'  var_smoothing (optimal): {best_vs:.2e}')
print(f'  Test accuracy:           {acc:.4f}  ({acc*100:.1f}%)')
print(f'  F1 (macro):              {f1_mac:.4f}')
print(f'  F1 (weighted):           {f1_wt:.4f}')
print('=' * 52)
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=axes[0], cmap='Greens', colorbar=False)
axes[0].set_title('Confusion matrix (Gaussian NB)')

cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': '% of true class'}, ax=axes[1])
axes[1].set_title('Normalised confusion matrix (%)')
plt.tight_layout(); plt.show()

# Per-class recall bar
recalls = [cm[i,i]/cm[i].sum()*100 for i in range(3)]
plt.figure(figsize=(6, 3))
bars = plt.bar(le.classes_, recalls, color=['#E8593C','#3B8BD4','#3DA87A'], edgecolor='white')
for b, v in zip(bars, recalls):
    plt.text(b.get_x()+b.get_width()/2, v+1, f'{v:.1f}%', ha='center', fontsize=10)
plt.axhline(100/3, linestyle='--', color='gray', linewidth=0.8, label='Random baseline (33%)')
plt.title('Per-class recall'); plt.ylabel('Recall (%)'); plt.ylim(0, 110)
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('Training set size before SMOTE:', len(y_train))
print('Training set size after SMOTE: ', len(y_train_sm))
print('Class distribution after SMOTE:', dict(zip(le.classes_, np.bincount(y_train_sm))))

gnb_sm = GaussianNB(var_smoothing=best_vs)
gnb_sm.fit(X_train_sm, y_train_sm)
y_pred_sm = gnb_sm.predict(X_test)

In [ ]:
def metrics(y_true, y_pred):
    return {
        'Accuracy':    accuracy_score(y_true, y_pred),
        'F1 macro':    f1_score(y_true, y_pred, average='macro'),
        'F1 weighted': f1_score(y_true, y_pred, average='weighted'),
    }

m_no = metrics(y_test, y_pred)
m_sm = metrics(y_test, y_pred_sm)

print('\n' + '='*52)
print('     SMOTE vs NO-SMOTE — METRIC COMPARISON')
print('='*52)
print(f"{'Metric':<18} {'No SMOTE':>12} {'+ SMOTE':>12} {'Change':>8}")
print('-'*52)
for key in m_no:
    diff = m_sm[key] - m_no[key]
    print(f"{key:<18} {m_no[key]:>12.4f} {m_sm[key]:>12.4f} {diff:>+8.4f}")
print('='*52)

In [ ]:
# Side-by-side normalised confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, title in zip(
    axes,
    [y_pred, y_pred_sm],
    ['Naive Bayes without SMOTE', 'Naive Bayes with SMOTE']
):
    cm = confusion_matrix(y_test, pred).astype(float)
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Greens',
                xticklabels=le.classes_, yticklabels=le.classes_,
                linewidths=0.5, linecolor='white',
                cbar_kws={'label': '% of true class'}, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.suptitle('Normalised confusion matrices: with vs without SMOTE', y=1.02)
plt.tight_layout(); plt.show()

# Per-class recall comparison bar chart
cm_no = confusion_matrix(y_test, y_pred)
cm_sm = confusion_matrix(y_test, y_pred_sm)
recalls_no = [cm_no[i,i]/cm_no[i].sum()*100 for i in range(3)]
recalls_sm = [cm_sm[i,i]/cm_sm[i].sum()*100 for i in range(3)]

x = np.arange(3); w = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x - w/2, recalls_no, w, label='No SMOTE', color='#B4B2A9', edgecolor='white')
b2 = ax.bar(x + w/2, recalls_sm, w, label='+ SMOTE',  color='#3DA87A', edgecolor='white')
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, f'{b.get_height():.1f}%',
            ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(le.classes_)
ax.set_ylabel('Recall (%)'); ax.set_ylim(0, 115)
ax.set_title('Per-class recall: no SMOTE vs + SMOTE')
ax.legend(); plt.tight_layout(); plt.show()

# Full classification reports
print('--- Without SMOTE ---')
print(classification_report(y_test, y_pred, target_names=le.classes_))
print('--- With SMOTE ---')
print(classification_report(y_test, y_pred_sm, target_names=le.classes_))

In [ ]:
# Download Model

import pickle
import os
from google.colab import files

os.makedirs('saved_models', exist_ok=True)

best_pred = y_pred_sm if m_sm['F1 macro'] >= m_no['F1 macro'] else y_pred
best_model = gnb_sm if m_sm['F1 macro'] >= m_no['F1 macro'] else gnb
best_label = '+ SMOTE' if m_sm['F1 macro'] >= m_no['F1 macro'] else 'No SMOTE'

print('=' * 52)
print('      NAIVE BAYES FINAL RESULTS SUMMARY')
print('=' * 52)
print(f'  var_smoothing:      {best_vs:.2e}')
print(f'  Best configuration: {best_label}')
print(f'  Dataset size:       {X.shape[0]} students')
print(f'  Features used:      {X_sel.shape[1]}')
print(f'  Accuracy:           {accuracy_score(y_test, best_pred):.4f}')
print(f'  F1 macro:           {f1_score(y_test, best_pred, average="macro"):.4f}')
print(f'  F1 weighted:        {f1_score(y_test, best_pred, average="weighted"):.4f}')
print('=' * 52)

nb_bundle = {
    'model':    best_model,
    'scaler':   scaler,
    'imputer':  imputer,
    'selector': sel,
    'encoder':  le,
}
with open('saved_models/nb_model.pkl', 'wb') as f:
    pickle.dump(nb_bundle, f)
print("Naive Bayes model saved -> saved_models/nb_model.pkl")

files.download('saved_models/nb_model.pkl')